<a href="https://colab.research.google.com/github/snr-laboratory/snrlab-ic-q-pix-v1/blob/chip_network_sim/chip_network_sim/dev_journals/kgosine_202603_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview


This program simulates an m x n chip network. Each chip can be decribed using only RTL and chips communicate via a lightweight messaging API. Each chip can recieve data from one upstream neighbor, send data to one downstream neighbor, and receives timing information from a shared launcher. Each chip also generates data locally such that data traffic is buffered using a single local FIFO. The routing of data through the network is defined via the Orchestrator launcher.

The system is a distributed synchronous hardware simualtor where there is a global clock, chips which act as hardware modules, and NNG sockets which act as wires between hardware modules.

## Chip Unit

A chip functions as a 2-input, 1-ouput unit. The first input is data from one upstream chip. The second input is data generated locally on the chip. The one output exits the chip's FIFO and is passed to the downstream chip.

The FIFO on the chip is defined with RTL (2-input, 1-output) which buffers data trafficed through the chip and acts as an arbiter between locally generated and pass-through data (data from upstream chip).

The locally generated data can be viewed as the output of any analog or mixed-signal front-end structure within the chip which generates data-packets. In this example, this front-end is generalized to a random number generator (defined in software) which generates data packets. The structure of a data packet is as follows:

* 64-bits long
* 16-bit chip_id
* 24-bit timestamp
* 24-bit data payload (randomly generated string)

The rate at which data is locally generated in a chip is configurable via a gen_ppm arguement. On every clock TICK, the chip generates a random number between 0-1,000,000 which is compared with the gen_ppm input. If the randomly generated number is less than the gen_ppm number, a data packet is created suring that timestep (otherwise no data packet is generated).  

A chip is defined primarily in the script chip_rtl.cpp which does the following:

* takes in the RTL (e.g. model.sv) describing some portion or all of the chip behavior
* run Verilator on the RTL (model.sv), converting it to C++ (model.cpp)
* communicates with the upstream chip to receive upstream data packet
* pass upstream data packet as input to the chip model (model.cpp)
* generate a local data packet and pass as an input to the chip model (model.cpp)
* simulate one complete time step of the chip model
* communicate with the downstream chip to pass output data packet
* communicate with Orchestrator to receive timestep TICK and send DONE reply
* send performance metrics to a metric collector script



#### Multithreading

A single chip unit can (and will, unless the chip is a sink chip) run two threads. The primary thread sends DATA_REQ to upstream chip (intiated by this chip and serialized with the TICK), getting the DATA_REP from upstream chip, run RTL timestep simulation, recieves TICK/DONE from orchestrator. The second thread handles downstream communication. This is triggered by the DATA_REQ from the downstream chip which is not necessarily serialized with the tick step and could possibly happen when the primary thread is busy doing its simualtion.

The two threads communicate via data_server_state_t which is shared state object between them. The primary thread publishes packet payloads to this data_server. The data-server thread will wait to reply to the downstream chip until the main thread has published to the data_server_state_t.

pthread is used to handle the synchronization. note the difference between this parallelization and our inter-chip network. pthread must work on the same process and share the same memory space whereas the nng socket commnication allows true disstributed computing across CPUs without shared memory.

## Config

run_from_config.py is an orchestrator launcher which passes arguments defined in a config.json to the Orchestrator (orchestrator.c). See network_3x4_snake_to_top_left.json for an example of how the routing, data packet, fifo_depth, and data packet generation can be set.


## Orchestrator

orchestrator.c is the top-level simualtion driver. it parses run settings (which can be passed using config.json) and then launches one chip process per grid cell. It is responsible for running the lock-step control protocol; it passes a TICK to all chips and waits to collect DONE from all chips, enforcing that all chips are stepping together in time. The orchestrator will use nng sockets for orchestrator-to-chp control/metrics to confirm time-stepping.



## Message Transport

The program uses NNG (nanomsg-next-gen) as a lightweight broker-less API to handle the movement of data packets between chips. There are three NNG channels defined in the program:

* REP socket: clock channel

Used for synchronization with the central clock driver provided by the orchestrator; sends each chip a TICK message which cues the chip simulation to run one cycle and reply with DONE message.

[ Clock (REQ) --TICK--> Chip (REP) ]

Each chip executes one simulation step

[ Chip (REP) --DONE--> Clock (REQ) ]

with added enforcement to WAIT until all chips reply DONE to send next tick, ensuring synchronization.


* REQ/REP socket: data channel between chips

Used for exchange of data packets between neighboring chips. The behavior depends if the chip is outputting or inputting data:

(a) If a chip is passing data downstream, REP server is set up. Waits for a DATA_PULL request from the downstream chip, then will respond with DATA_REPLY. In short, the downstream chip pulls data for at a specific TICK and the server answers when the data at that TICK is ready. If the chip did not produce an output packet at that tick, it will send nothing.

[ Downstream Chip (REQ) --DATA_PULL-->  Chip (REP) ]

[ Chip (REP) --DATA_REPLY--> Downstream Chip (REQ) ]

(b) If the chip is reading from upstream is opens a REQ socket. Each tick it will request a data packet from upstream, wait for a reply from the upstream chip, then validates that the data packet which arrived has the correct upstream chip id.

[ Chip (REQ) --DATA_REQUEST--> Upstream Chip (REP) ]

[ Upstream Chip (REP) --DATA_REPLY--> Chip (REQ) ]

* PUSH socket: metrics channel which sends one-way reporting to a metrics colelctor

No blocking or wait for reponse. Used to gather metrics about the simulation.


On each TICK:
* The orchestrator sends TICK(seq) over the REP socket (chips recieve)
* A downstream chip dends DATA_PULL request (REQ socket). The upstream chip's data_server_tread_main() receives the pull requeston its REP socket. It then sends a data packet (should one already be published by the main simulator) over its REP socket to the REQ socket of the downstream chip.

Note that requests made all have the specifi sequence (tick) number. This ensures that chips which are simulating at different speeds do not leak duture information backwards in time. eg if Chip A is simualting faster than Chip B, Chip B could request the data packet for its current tick number (10). If it were to request simply the latest packet from Chip A, which is already on tick 12, it could receive a packet from cycle 12 when it should have received for cycle 10. Therefore, we ensure that requests specify the cycle number, accounting for differences in chip simualtion times across CPUs.

## Future Directions


* incorporating a Spice (analog) portion into the simulation

we've previously successfully performed Spice/RTL cosimulation using Verilator before

* allowing data input from any of the four adjacent chips

this will allow for configuration routes beyond daisy chaining which is obviously vulnerable to single point failure. at this stage, the routing would still be defined at the top level.

* incorpoarting existing RTL (LArPix) including more realistic data packet creation and transmission timing.



## Applications

* because of the distributed network design of the architecture, chips on the order of those needed in DUNE: for 4x4mm pixels, with 62,500 pixels per m^2. Assuming 64 channel chips, on the order of 10^3 chips/ m^2. Assuming pixelating just one face of a DUNE module, that's 380 m^2 so somewhere around 380,000 chips. the only way to do a simualtion of this scale is with distributed networking.

* taking a design from RTL/Spice (transistor-level) to a tile-level system reduces the number of edge cases which are missed. No part of the simualtion path from chip to tile is simplified, increasing confidence of final product. any unintended effects from the transistor-level or communication design wil appear in the simulation results.

* modifiable to any generic chip: highlighted with this toy model (and later by incorporating analog components), this architecture allows for plugging in any generic chip design or chip routing scheme. Further, the communication system can be tweaked for ACK/NACK signals to be included allowing for testing of new networking schemes with minimal changes to the software. The chip design needs only to be compiled with Verilator once and then can quickly be used in a tile simualtion.

* simple simualtions can show packet loss as a function of FIFO sizes, maximum data generation rates, routing schemes ect.



## Logic Elements

(1) Backpressure via the global clock barrier

Because a chip cannot send DONE until it has finished its simulation tick, anything that gets held up during the TICK pushes back on the whole system. One slow chip slows the entire simulation.

(2) Backpressure on the pull-based data transfer

The downstream chip clint requests data for a specific tick. The simulation is stalled until the upstream chip as simulated and published data for that sequence. If the upstream chip is running behind, the server thread is blocked.

(3) Dropping implemented in the FIFO. If the FIFO is full, incoming packets are dropped.

(4) Backpressure on pass-through packet generation. Locally generated packets take priority for FIFO entry. Therefore, pass through packets will 'hold' and wait to enter the FIFO after the local packet has entered (reducing availability by 1). If both are trying to get into the FIFO at the same time and there is 2 or more available positions in the FIFO, both will enter on the SAME tick. Therefore, arrival of both local and pass through packets at the same time will not cause a data loss unless the FIFO has 0 or 1 position available.

## Test Cases


###(1) Lockstep barrier

The transport is pull-based but storage is single-slot. Correctness relies on seq matching + lockstep. If lockstep breaks, there is no buffering to recover old packets. eg if Chip B (slow) is still working on tick N (hasn't sent DONE) and chip A has published for tick N but then B requests for a tick that doesn't match what Chip A has retained. This would indicate a failure in the lockstep mechanism.

Possible test: artificially slow the downstream so it requests "old" seq. If the clocking barrier is strict this should just slow down the global simualtion.



###(2) Determinism

With the same seed, run the same simulation 10-20 times, measure total delivered packets, drops per chip, end-to-end latency and enture identical results every run.

#### Determinism Test Report (3x5 Snake BR->TL)

- Config: `/home/lxusers/k/kalindigosine/snrlab-ic-q-pix-v1/chip_network_sim/config/network_3x5_determinism_snake_br_to_tl.json`
- Runs: 15
- Deterministic behavior check (`delivered_tx + total_drops + per_chip_drops`): **PASS**
- Full output check (including `cycles_per_sec`): **FAIL**

##### Per-run Results

| Run | Delivered Packets (tx) | Total Drops | Cycles/sec | Per-chip Drops (chip_id:count) | Behavior Match vs Run1 | Full Match vs Run1 |
| --- | ---: | ---: | ---: | --- | --- | --- |
| 1 | 261827 | 0 | 2329.463 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | yes |
| 2 | 261827 | 0 | 2266.484 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 3 | 261827 | 0 | 2304.796 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 4 | 261827 | 0 | 2283.983 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 5 | 261827 | 0 | 2245.862 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 6 | 261827 | 0 | 2291.448 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 7 | 261827 | 0 | 2266.732 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 8 | 261827 | 0 | 2192.259 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 9 | 261827 | 0 | 2233.954 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 10 | 261827 | 0 | 2211.480 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 11 | 261827 | 0 | 2203.634 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 12 | 261827 | 0 | 2131.800 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 13 | 261827 | 0 | 2142.861 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 14 | 261827 | 0 | 2169.355 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 15 | 261827 | 0 | 2166.838 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |

The fastest/slowest runs were 2329.463/2131.800 cycles/sec. Runtime speed varied by ~2.7%, typical for CPU noise.

### (3) FIFO arbitration correctness for local priority under near-full conditions

2-chip link where the downstream chip is DUT. Set gen_ppm to a large value. Set small fifo_depth (1 or 2). Measure local packets dropped versus pass through packets dropped.

#### 1x2 Local-Priority Packet-Loss Test Report

- Config: `/home/lxusers/k/kalindigosine/snrlab-ic-q-pix-v1/chip_network_sim/config/network_1x2_priority_loss_test.json`
- Trace run: `traces/priority_loss_1x2/run_1772650434_2818623`
- Run log: `/home/lxusers/k/kalindigosine/snrlab-ic-q-pix-v1/chip_network_sim/reports/priority_loss_1x2/latest_run.log`

##### Test Setup

- Topology: chip `0 -> 1`, with chip `1` as downstream sink (`out_id=-1`)
- Grid: `1x2` chips
- `gen_ppm`: `1,000,000` for both chips (generate every tick)
- `fifo_depth`: `2`
- `ticks`: `2000`

##### Run-Level Metrics (orchestrator)

- delivered packets (`tx`): 1999
- received packets (`rx`): 1999
- locally generated packets (`local`): 4000
- total drops (`drops`): 1998
- fifo peak occupancy (`fifo_peak`): 2
- cycles/sec: 5220.575

##### Packet Loss by Chip (from trace events)

| Chip | Local Enq OK | Neighbor Enq OK | Local Drops | Neighbor Drops | Total Drops | DEQ_OUT |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 2000 | 0 | 0 | 0 | 0 | 1999 |
| 1 | 2000 | 1 | 0 | 1998 | 1998 | 1999 |

##### Sink Output Provenance (chip 1, DEQ_OUT events)

- Total sink output packets: 1999
- From chip 1 local traffic (`src_id=1`): 1998
- From chip 0 pass-through traffic (`src_id=0`): 1
- `src_id=0` packet `timestamp` values: `0`
- From other sources: 0

##### Finding vs Expectation

- Result: PARTIAL/FAIL for the strict expectation.
- Observation: sink output includes at least one pass-through packet from chip 0.
- Interpretation: local-first priority does not block neighbor packets when capacity is available; with depth=2, one neighbor packet can still enter and later dequeue.

### FIFO Queuing

Track FIFO occupancy for a single chip in a run. In this example, the 2x2 network was used. Chip 3 sends traffic into Chip 2 which is also generating traffic at a rate sufficient to fill the FIFO and begin dropping packets. The FIFO is 5-wide and the Queue is shown below:



| tick | slot_0 | slot_1 | slot_2 | slot_3 | slot_4 |
| --- | --- | --- | --- | --- | --- |
| 0 | 0x0002000000CFCF6B | - | - | - | - |
| 1 | 0x0002000001825438 | 0x000300000044B65A | - | - | - |
| 2 | 0x000300000044B65A | 0x00020000023248A7 | 0x00030000015FA625 | - | - |
| 3 | 0x00020000023248A7 | 0x00030000015FA625 | 0x00020000033C96B2 | 0x0003000002315EFF | - |
| 4 | 0x00030000015FA625 | 0x00020000033C96B2 | 0x0003000002315EFF | 0x00020000043CC553 | 0x0003000003A8A8C6 |
| 5 | 0x00020000033C96B2 | 0x0003000002315EFF | 0x00020000043CC553 | 0x0003000003A8A8C6 | 0x00020000058C00CC |
| 6 | 0x0003000002315EFF | 0x00020000043CC553 | 0x0003000003A8A8C6 | 0x00020000058C00CC | 0x0002000006042E9C |
| 7 | 0x00020000043CC553 | 0x0003000003A8A8C6 | 0x00020000058C00CC | 0x0002000006042E9C | 0x0002000007F57D00 |
| 8 | 0x0003000003A8A8C6 | 0x00020000058C00CC | 0x0002000006042E9C | 0x0002000007F57D00 | 0x00020000082A26B5 |
| 9 | 0x00020000058C00CC | 0x0002000006042E9C | 0x0002000007F57D00 | 0x00020000082A26B5 | 0x0002000009EA2C3A |
| 10 | 0x0002000006042E9C | 0x0002000007F57D00 | 0x00020000082A26B5 | 0x0002000009EA2C3A | 0x000200000A9932CD |

###(4) Congestion wave in a network

Uniform into each chip. For a chained network, should see congestion propagate through the network to chips downstream.

## Packet Tracing in 2x2 Network



![](https://drive.google.com/uc?export=view&id=11gUIXXqPOtzLXKPXAkTtgh4XxxuuSGR_)

## Simulation Metrics


cycles/sec run time ect